In [ ]:
%pip install -q fuzzyclust


# Gath–Geva: clusters of different size and density

GK fixes the volume of every cluster, so it handles a small dense cluster next
to a large diffuse one poorly: the volume constraint forces the dense cluster
to "steal" points from its big neighbor. The Gath–Geva (GG) algorithm uses
the fuzzy maximum-likelihood distance

\begin{align}D^2_{ik} = \frac{\sqrt{\det\mathbf{F}_{wi}}}{\alpha_i}
    \exp\!\left(\tfrac12(\mathbf{x}_k-\mathbf{v}_i)^T\mathbf{F}_{wi}^{-1}(\mathbf{x}_k-\mathbf{v}_i)\right),\end{align}

the reciprocal of a Gaussian density weighted by the prior $\alpha_i$.
Both the volume **and** the density of each cluster adapt, so GG recovers
clusters of strongly different size. The price is sensitivity to
initialization: with the exponential distance the iteration quickly
converges to a nearby local optimum, so GG is almost always initialized with
the partition of FCM or GK.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import adjusted_rand_score

from fuzzyclust import FCM, GG, GK
from fuzzyclust.datasets import make_different_densities
from fuzzyclust.plotting import plot_covariance_ellipses, plot_membership_contours

X, y_true = make_different_densities(sizes=(300, 60, 60), spreads=(1.5, 0.3, 0.6), random_state=0)

fcm = FCM(n_clusters=3, random_state=0).fit(X)
gk = GK(n_clusters=3, random_state=0).fit(X)
gg = GG(n_clusters=3, init=fcm.membership_).fit(X)  # initialized from FCM

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharex=True, sharey=True)
for ax, model, name in zip(axes, (fcm, gk, gg), ("FCM", "GK", "GG (init: FCM)"), strict=False):
    plot_membership_contours(model, X, ax=ax, labels=False)
    plot_covariance_ellipses(model.cluster_centers_, model.covariances_, ax=ax)
    ax.set_title(f"{name}: ARI = {adjusted_rand_score(y_true, model.labels_):.2f}")
plt.tight_layout()

FCM and GK let the two small clusters take over the margin of the big
one; GG assigns it a large covariance and a large prior probability and
leaves the small dense clusters intact. The
priors $\alpha_i=\frac1N\sum_k\mu_{ik}$ estimate the relative sizes:



In [ ]:
print("GG prior probabilities:", gg.priors_.round(3))
print("true proportions      :", (np.bincount(y_true) / len(y_true)).round(3))

## Why initialization matters
Started from random prototypes, GG can end in a poor local optimum, and the
outcome varies from seed to seed. We run GG from ten random initializations
and from the FCM partition and compare the agreement with the generating
labels: the FCM-initialized run is as good as the best random run, without
depending on the seed.



In [ ]:
aris = []
for seed in range(10):
    model = GG(n_clusters=3, init="random", random_state=seed).fit(X)
    aris.append(adjusted_rand_score(y_true, model.labels_))
print(f"random init  : ARI mean = {np.mean(aris):.2f}, min = {np.min(aris):.2f}, max = {np.max(aris):.2f}")
print(f"FCM init     : ARI = {adjusted_rand_score(y_true, gg.labels_):.2f}")

## The GG distance interpreted as a density
Because $1/D^2_{ik}$ is proportional to a Gaussian density, the
membership $\mu_{ik}$ is the posterior probability that
$\mathbf{x}_k$ was generated by component $i$. The figure shows
the sum of $\alpha_i\,\mathcal{N}(\mathbf{x};\mathbf{v}_i,\mathbf{F}_i)$
over the three fitted components.



In [ ]:
gx, gy = np.meshgrid(np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 200),
                     np.linspace(X[:, 1].min() - 1, X[:, 1].max() + 1, 200))
G = np.c_[gx.ravel(), gy.ravel()]
dens = np.zeros(G.shape[0])
for i in range(3):
    diff = G - gg.cluster_centers_[i]
    inv = np.linalg.inv(gg.covariances_[i])
    maha = np.einsum("kj,jl,kl->k", diff, inv, diff)
    dens += gg.priors_[i] * np.exp(-0.5 * maha) / (2 * np.pi * np.sqrt(np.linalg.det(gg.covariances_[i])))
fig, ax = plt.subplots(figsize=(5.5, 4.5))
cf = ax.contourf(gx, gy, dens.reshape(gx.shape), levels=20, cmap="Blues")
ax.scatter(X[:, 0], X[:, 1], s=5, color="k", alpha=0.5)
plt.colorbar(cf, ax=ax, label="mixture density")
ax.set_title("GG clusters as a Gaussian mixture")
plt.tight_layout()

## Parameter study: the fuzziness exponent for FCM, GK, and GG
The exponent $m$ is the one parameter the three algorithms share.
Scanning it on the same data separates two effects: the softness of the
partition (set by $m$) and the shape of the distance (set by the
algorithm). The agreement with the generating labels and the GG priors are
recorded for every $m$; GG is initialized from the FCM partition
obtained with the same exponent.



In [ ]:
from fuzzyclust.plotting import PALETTE

# [snippet:m_scan]
m_values = (1.2, 1.5, 2.0, 2.5, 3.0)
ari = {"FCM": [], "GK": [], "GG (init: FCM)": []}
priors = []
for m in m_values:
    fcm_m = FCM(n_clusters=3, m=m, random_state=0).fit(X)
    gk_m = GK(n_clusters=3, m=m, random_state=0).fit(X)
    gg_m = GG(n_clusters=3, m=m, init=fcm_m.membership_).fit(X)
    for name, model in zip(ari, (fcm_m, gk_m, gg_m), strict=True):
        ari[name].append(adjusted_rand_score(y_true, model.labels_))
    priors.append(np.sort(gg_m.priors_)[::-1])
# [/snippet:m_scan]

fig, ax = plt.subplots(figsize=(6.5, 4))
for j, (name, values) in enumerate(ari.items()):
    ax.plot(m_values, values, "o-", color=PALETTE[j], label=name)
ax.set_xlabel("fuzziness exponent $m$")
ax.set_ylabel("ARI with the generating labels")
ax.set_ylim(0, 1.05)
ax.set_title("the distance, not the exponent, decides whether the dense clusters survive")
ax.legend(frameon=False)
ax.grid(True, color="0.9")
plt.tight_layout()

print("  m    FCM    GK    GG     GG priors (sorted)")
for j, (m, pr) in enumerate(zip(m_values, priors, strict=True)):
    print(f"{m:4.1f}  {ari['FCM'][j]:5.2f}  {ari['GK'][j]:5.2f}  {ari['GG (init: FCM)'][j]:5.2f}   {pr.round(2)}")

## Take-away
The exponential distance of Gath--Geva makes the cluster volume and the
prior probability part of the model, so a large diffuse group and small
dense groups can coexist in one partition, and the memberships are the
posterior probabilities of a Gaussian mixture. FCM and GK, whose distances
fix the volume, cut into the large group whatever the exponent: changing
$m$ alters how soft the partition is but not which structure it
describes, so across the whole range GG stays near $ARI\approx1$
while FCM and GK stay far below it. The price of the flexible distance is
a rugged objective; as the restart experiment above shows, GG should be
started from a converged FCM or GK partition rather than from random
prototypes.

